Контрольне питання №1 — чому t, а не z?
Відповідь:

Для перевірки середнього однієї вибірки використовується t-розподіл, оскільки генеральне стандартне відхилення σ невідоме. Замість нього використовується вибіркове стандартне відхилення s, яке саме оцінюється за даними. t-розподіл враховує додаткову невизначеність, пов'язану з оцінюванням σ, тому для невідомого σ використовується t-критерій.

Контрольне питання №2 — навіщо Лівен?
Питання:

Чому неправильний equal_var може дати систематично неправильний p-value?

Відповідь:

Параметр equal_var визначає, яку формулу двовибіркового t-критерію використовувати. Стандартний t-критерій припускає рівність дисперсій, тоді як Welch-критерій не потребує такого припущення. Якщо при нерівних дисперсіях помилково використати equal_var=True, стандартна формула може неправильно оцінити стандартну похибку та розподіл статистики, а отже, отриманий p-value може бути систематично спотвореним, а не просто трохи менш точним. Тому перед вибором варіанта t-критерію у цій роботі спочатку застосовується критерій Лівена.

Контрольне питання №3 — чому парний критерій потужніший?

Парний критерій може бути потужнішим за незалежний двовибірковий критерій для пов'язаних даних, оскільки аналізує різниці всередині кожної пари. При цьому усувається частина варіації, яка є спільною для двох вимірювань однієї пари. У випадку ранкової та вечірньої вологості порівнюються вимірювання одного й того самого місяця, тому сезонні особливості місяця не розглядаються як окремий шум при аналізі різниці.

Контрольне питання №4 — Лівен проти .std()


.std() просто показує:

наскільки сильно розкидані дані.

А levene() формально перевіряє гіпотезу про рівність дисперсій.

Порівняння .std() дозволяє приблизно оцінити відмінність розкиду двох вибірок наочно, але не є формальним статистичним тестом. Критерій Лівена дає формальний p-value для перевірки гіпотези про рівність дисперсій. Тому в роботі рішення щодо equal_var потрібно приймати саме на основі levene(), а не лише за візуальним порівнянням стандартних відхилень.

Для порівняння ранкової та вечірньої вологості використано парний t-критерій ttest_rel, оскільки кожне ранкове значення безпосередньо відповідає вечірньому значенню того самого місяця. Це не дві незалежні вибірки, а 12 пов'язаних пар спостережень. Парний критерій фактично аналізує різниці між значеннями всередині кожної пари, завдяки чому усувається вплив спільних для пари факторів, зокрема особливостей конкретного місяця. Тому для таких даних парний критерій є коректнішим за незалежний двовибірковий критерій.

In [28]:
differences = humidity - evening

print("Різниці ранок - вечір:")
print(differences)

print("Середня різниця =", differences.mean())
print("Стандартне відхилення різниць =", differences.std(ddof=1))


Різниці ранок - вечір:
[3 3 4 5 6 7 7 7 6 5 4 3]
Середня різниця = 5.0
Стандартне відхилення різниць = 1.5954480704349312


In [27]:
t_rel, p_rel = stats.ttest_rel(
    humidity,
    evening
)

print("t =", t_rel)
print("p-value =", p_rel)


t = 10.85620296683619
p-value = 3.2341040311217473e-07


In [26]:
evening = humidity - amplitude

print("Ранок:")
print(humidity)

print("\nВечір:")
print(evening)


Ранок:
[84 82 78 70 65 63 61 63 68 75 81 85]

Вечір:
[81 79 74 65 59 56 54 56 62 70 77 82]


In [25]:
amplitude = np.array([
    3, 3, 4, 5, 6, 7,
    7, 7, 6, 5, 4, 3
])


Для другого порівняння було обрано місто ..., яке належить до ... регіону. На відміну від Завдання 3, критерій Лівена дав протилежний висновок щодо рівності дисперсій: p = ... . Тому тут використано [стандартний / Welch] варіант ttest_ind, тобто equal_var=True/False. Вибір варіанта t-критерію зроблено на основі результату критерію Лівена, а не на основі припущення чи візуального порівняння даних.

In [24]:
t_ind_2, p_ind_2 = stats.ttest_ind(
    humidity,
    data2,
    equal_var=equal_var_2
)

print("t =", t_ind_2)
print("p-value =", p_ind_2)


t = 0.5956306865009461
p-value = 0.557500478851693


In [23]:
if lev_p_2 < 0.05:
    equal_var_2 = False
    print("Використовуємо Welch t-test.")
else:
    equal_var_2 = True
    print("Використовуємо стандартний t-test.")


Використовуємо стандартний t-test.


In [22]:
lev_stat_2, lev_p_2 = stats.levene(
    humidity,
    data2
)

print("Місто:", city2)
print("Levene statistic =", lev_stat_2)
print("Levene p-value =", lev_p_2)



Місто: Харків
Levene statistic = 0.07624064319378997
Levene p-value = 0.7850353624913209


In [21]:
city2 = "Харків"
data2 = cities[city2]


In [16]:
reference_city = "Львів"

reference_stat, reference_p = stats.levene(
    humidity,
    cities[reference_city]
)

reference_different = reference_p < 0.05

print("Львів:")
print("Levene p =", reference_p)

for city, data in cities.items():
    if city == reference_city:
        continue
    
    if regions[city] == "Центр":
        continue
    
    _, p = stats.levene(humidity, data)
    
    city_different = p < 0.05
    
    if city_different != reference_different:
        print("\nЗнайдено місто з протилежним результатом:")
        print(city)
        print("Регіон:", regions[city])
        print("Levene p =", p)
        break


Львів:
Levene p = 0.7229054121343448

Знайдено місто з протилежним результатом:
Одеса
Регіон: Південь
Levene p = 0.0197605527986236


In [15]:
for city, data in cities.items():
    if regions[city] != "Центр":
        lev_stat_city, lev_p_city = stats.levene(
            humidity,
            data
        )
        
        print(
            f"{city:20s} | "
            f"Levene p = {lev_p_city:.4f}"
        )


Львів                | Levene p = 0.7229
Одеса                | Levene p = 0.0198
Харків               | Levene p = 0.7850
Дніпро               | Levene p = 0.7850
Івано-Франківськ     | Levene p = 0.4801
Суми                 | Levene p = 0.7850
Ужгород              | Levene p = 0.3915


In [14]:
regions = {
    "Львів": "Захід",
    "Одеса": "Південь",
    "Харків": "Схід",
    "Дніпро": "Схід",
    "Чернігів": "Центр",
    "Івано-Франківськ": "Захід",
    "Полтава": "Центр",
    "Суми": "Схід",
    "Ужгород": "Захід"
}


In [13]:
cities = {
    "Львів": np.array([85, 83, 80, 73, 68, 65, 63, 65, 70, 77, 82, 86]),
    "Одеса": np.array([82, 80, 78, 74, 71, 69, 67, 68, 72, 76, 80, 83]),
    "Харків": np.array([83, 80, 75, 68, 62, 60, 58, 60, 66, 73, 79, 84]),
    "Дніпро": np.array([82, 79, 74, 67, 61, 59, 57, 59, 65, 72, 78, 83]),
    "Чернігів": np.array([85, 83, 79, 71, 66, 63, 61, 63, 69, 76, 82, 86]),
    "Івано-Франківськ": np.array([86, 84, 81, 75, 70, 67, 65, 67, 72, 78, 83, 87]),
    "Полтава": np.array([84, 81, 76, 69, 63, 61, 59, 61, 67, 74, 80, 85]),
    "Суми": np.array([85, 82, 77, 70, 64, 62, 60, 62, 68, 75, 81, 86]),
    "Ужгород": np.array([84, 82, 79, 73, 68, 66, 64, 66, 71, 77, 82, 85])
}


За критерієм Лівена отримано p = ... . Оскільки p ... 0.05, використовуємо [стандартний / Welch] варіант двовибіркового t-критерію. За результатом t-критерію p = ... . При α = 0.05 [відхиляємо / не відхиляємо] H0. Отже, [є / немає] статистично значущих підстав вважати, що середня вологість Києва відрізняється від середньої вологості Львова.

In [12]:
t_ind, p_ind = stats.ttest_ind(
    humidity,
    lviv,
    equal_var=equal_var
)

print("t =", t_ind)
print("p-value =", p_ind)


t = -0.5161780659968352
p-value = 0.6108795276254253


In [11]:
if lev_p < 0.05:
    equal_var = False
    print("Дисперсії вважаємо статистично різними.")
    print("Використовуємо Welch t-test.")
else:
    equal_var = True
    print("Немає підстав вважати дисперсії різними.")
    print("Використовуємо стандартний t-test.")


Немає підстав вважати дисперсії різними.
Використовуємо стандартний t-test.


In [10]:
lev_stat, lev_p = stats.levene(
    humidity,
    lviv
)

print("Levene statistic =", lev_stat)
print("Levene p-value =", lev_p)


Levene statistic = 0.12899022801302928
Levene p-value = 0.7229054121343448


In [9]:
lviv = np.array([
    85, 83, 80, 73, 68, 65,
    63, 65, 70, 77, 82, 86
])

print("Київ:", humidity)
print("Львів:", lviv)


Київ: [84 82 78 70 65 63 61 63 68 75 81 85]
Львів: [85 83 80 73 68 65 63 65 70 77 82 86]


t-статистика була додатково обчислена вручну за формулою 
t
=
(
x
ˉ
−
μ
0
)
/
(
s
/
n
)
. Отримане значення збігається з результатом stats.ttest_1samp() з точністю до округлення. Це підтверджує, що функція ttest_1samp використовує ту саму формулу t-статистики.

In [8]:
print(
    "Збігаються:",
    np.isclose(t_manual, t_stat)
)


Збігаються: True


In [7]:
t_manual = (x_bar - norm) / se

print("t вручну =", t_manual)
print("t scipy =", t_stat)


t вручну = -0.4193548387096755
t scipy = -0.41935483870967566


In [6]:
se = s / np.sqrt(n)

print("SE =", se)



SE = 2.583333333333334


In [5]:
alpha = 0.05

if p_value < alpha:
    print("Відхиляємо H0.")
else:
    print("Не відхиляємо H0.")


Не відхиляємо H0.


In [4]:
t_stat, p_value = stats.ttest_1samp(
    humidity,
    popmean=norm
)

print("t =", t_stat)
print("p-value =", p_value)


t = -0.41935483870967566
p-value = 0.6830295031131001


In [3]:
x_bar = humidity.mean()
s = humidity.std(ddof=1)
n = len(humidity)

print("Середнє x̄ =", x_bar)
print("Стандартне відхилення s =", s)
print("n =", n)


Середнє x̄ = 72.91666666666667
Стандартне відхилення s = 8.9489291724392
n = 12


H0: μ = 74% — середня вологість відповідає довідковій нормі.

H1: μ ≠ 74% — середня вологість статистично відрізняється від довідкової норми.

Використовується двобічний тест, оскільки нас цікавить будь-яке відхилення від норми, а не лише збільшення або лише зменшення вологості.

In [2]:
humidity = np.array([
    84, 82, 78, 70, 65, 63,
    61, 63, 68, 75, 81, 85
])

norm = 74

print("Місячні значення вологості:")
print(humidity)

print("Норма =", norm)


Місячні значення вологості:
[84 82 78 70 65 63 61 63 68 75 81 85]
Норма = 74


In [1]:
import numpy as np
from scipy import stats


# Практика 14. Параметричні критерії

**Варіант:** 1  
**Місто:** Київ  
**Регіон:** Центр  
**Норма вологості:** 74%
